# flatbug Training & Inference — 22-Class Insect Detection

Fine-tunes the **flatbug** arthropod model (https://darsa.info/flat-bug/) on the
synthetic tray dataset `insect_dataset_mixed`, with **before-training** looks
at the raw pretrained model (synthetic + real pitfall data) and **after-training**
evaluation + inference.

**How this works (read me):**
flatbug ships as a *single-class* arthropod **segmentation** model (a YOLOv8-seg
checkpoint, e.g. `flat_bug_L.pt`). Our dataset is **22-class bounding-box** COCO with
**no masks**, so we can't train flatbug's native single-class seg head on it.

Instead we do the thing that actually transfers flatbug's value: we **warm-start a
YOLOv8 *detection* model from flatbug's arthropod-tuned backbone + neck**, give it a
fresh 22-class detection head, and fine-tune on our bboxes. This keeps the run a
direct apples-to-apples comparison with the RF-DETR run (same task, same dataset,
same mAP metric) while standing on flatbug's pretrained arthropod features.

**Workflow:**
1. Install dependencies
2. Configure all settings (edit before every run)
3. Download the flatbug weights
4. **Baseline (before training)** — pretrained flatbug on synthetic test trays
5. **Real-data config** — point at `3DTRAB_PITFALL_TRAP_Dataset_v1`
6. **Real-data PRE inference (before training)** — pretrained flatbug on the real trap images
7. Convert the COCO dataset → YOLO detection format
8. Sanity-check the converted dataset
9. Build the model & warm-start from flatbug
10. Train
11. Evaluate on the held-out test split (per-class mAP)
12. **Post-training inference** (synthetic test) — visualise + write `detections.csv`
13. **Real-data POST inference (after training)** — fine-tuned model on the real trap images
14. Export `best.pt` → ONNX

---
**Requirements before running:**
- `insect_dataset_mixed/` with `train/`, `valid/`, `test/` sub-folders, each
  containing images + `_annotations.coco.json` (Roboflow-style COCO).
- A CUDA GPU (written for the Ares single-GPU box).


---
## Cell 1 — Install Dependencies

flatbug is built on **Ultralytics YOLOv8**. Training uses plain `ultralytics`;
the pretrained baselines optionally use flatbug's own tiled `Predictor`, so we also
install the local `flat-bug-1.0.0` package. `ultralytics` is pinned to the range
flatbug supports. Run once per environment, then restart the kernel.

> Install a CUDA-matched `torch` first if it isn't already (see https://pytorch.org).
> flatbug's package needs Python ≥ 3.11; if that install fails, the baseline cells
> automatically fall back to plain Ultralytics seg-predict.
>
> **Headless-server note:** `ultralytics` depends on `opencv-python`, which needs
> `libGL` and errors with `libGL.so.1: cannot open shared object file` on a headless
> box. This cell forces the headless build (pinned to 4.10.0.84, what roboflow/RF-DETR
> in this shared env require) so `import cv2` works.


In [ ]:
import subprocess, sys
from pathlib import Path

# Ultralytics version compatible with BOTH training and flatbug's native Predictor.
subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics>=8.2.16,<=8.3.124"])

# ultralytics pulls in non-headless opencv-python, which needs libGL and crashes
# `import cv2` on headless servers. Remove ALL opencv builds, then install the
# headless build PINNED to 4.10.0.84 (what roboflow/RF-DETR in this shared env
# require, and a known-good version for ultralytics + flatbug). Uninstalling a
# missing package is a harmless skip.
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
                       "opencv-python", "opencv-contrib-python",
                       "opencv-python-headless", "opencv-contrib-python-headless"])
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "opencv-python-headless==4.10.0.84"])

# (Optional) local flatbug package -> enables flatbug's native tiled baselines.
_fb = Path("flat-bug-1.0.0")
if _fb.exists():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(_fb)])
        print("flat_bug (editable) installed - native baseline available.")
    except subprocess.CalledProcessError as e:
        print(f"flat_bug editable install failed ({e}); baselines will use the Ultralytics fallback.")
else:
    print("flat-bug-1.0.0/ not found; baselines will use the Ultralytics fallback.")

print("\nInstallation complete.")
print("IMPORTANT: Restart the kernel now (Kernel -> Restart), then run from Cell 2 onwards.")


---
## Cell 2 — Configuration

> **This is the main section to edit before every run.** All knobs live here.


In [ ]:
# ============================================================
# SEPTEMBER RUN — flatbug, whole tray @1536
# ============================================================
# Regenerated dataset, trained at full resolution. What changed since August:
#
#   * GROUND TRUTH boxes are the insect, not the rotated canvas. The old boxes were
#     1.295x too large on a side (4/pi for a uniform angle), so ~40% of every box was
#     tray. Every AP number from before was measured against inflated targets.
#   * TRAYS are 3473x2769 = 10.85 px/mm, matching the real 3DTRAB trays exactly.
#     Previously 2644x2108 = 8.26 px/mm, so synthetic insects were 24% too small.
#   * TAXONOMY is 22 classes, not 23. Siphonaptera and Raphidioptera dropped;
#     Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
#   * DATA is synthetic + real 3DTRAB mixed, group-disjoint on the real side, with
#     the real half oversampled to ~5% of the training signal.
#
# Evaluate on 3dtrab_coco_TESTSPLIT.json, NEVER the full 3DTRAB — roughly half of
# those photographs are in this training set.

from pathlib import Path
import os

# ─────────────────────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))   # <-- Ares path; matches final_comparison

# Roboflow-style COCO dataset (train/ valid/ test/, each with _annotations.coco.json).
COCO_DATASET_DIR = BASE_DIR / "insect_dataset_mixed"      # synthetic + real, mixed

# Where the YOLO-format copy of the dataset will be written (images symlinked, labels generated).
YOLO_DATASET_DIR = BASE_DIR / "sept_run" / "flatbug_yolo_mixed"

# Where checkpoints / logs go.  Ultralytics creates OUTPUT_DIR/RUN_NAME/.
OUTPUT_DIR = BASE_DIR / "sept_run" / "flatbug_whole_1536"
RUN_NAME   = "flatbug_l_1536"

# ─────────────────────────────────────────────────────────────────────────────
# flatbug WEIGHTS  ← the pretrained arthropod model we warm-start from
# ─────────────────────────────────────────────────────────────────────────────
# Size: "S" (yolov8s), "M" (yolov8m, recommended), or "L" (yolov8l, best/heaviest).
FLATBUG_SIZE    = "L"
FLATBUG_WEIGHTS = BASE_DIR / f"flat_bug_{FLATBUG_SIZE}.pt"   # auto-downloaded in Cell 3 if missing
# flatbug's public weight store (from flat_bug/__init__.py REMOTE_REPOSITORY):
FLATBUG_REMOTE  = "https://anon.erda.au.dk/share_redirect/Bb0CR1FHG6/models/"

# Matching Ultralytics detection architecture to warm-start into.
BASE_ARCH = f"yolov8{FLATBUG_SIZE.lower()}.pt"   # COCO-pretrained detect model; head reset to 22 classes at train

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING HYPER-PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
DEVICE   = "1"          # 0=RF-DETR whole, 1=flatbug whole, 2=RF-DETR tiled, 3=flatbug tiled
IMG_SIZE = 1536         # trays are 3473 px wide, so this feeds them at 0.44x and a
                        # real beetle lands at ~58 px. At 1024 it was 39 px, close to
                        # COCO's 32 px "small object" line. Multiple of 32.
EPOCHS   = 100
BATCH    = 4            # 1536 is 2.25x the pixels of 1024; raise to 8 if VRAM allows
WORKERS  = 8
PATIENCE = 20           # early-stopping patience (matches the RF-DETR run)
SEED     = 42

# Optimiser / LR schedule (YOLOv8 fine-tune defaults; flatbug's best_train used SGD lr0=1e-3, lrf=1e-5).
OPTIMIZER     = "SGD"
LR0           = 0.01
LRF           = 0.01    # final LR = LR0 * LRF (cosine)
MOMENTUM      = 0.937
WEIGHT_DECAY  = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC  = 10      # disable mosaic aug for the last N epochs (helps dense small-object trays)

# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
CONF_THRES = 0.4        # confidence threshold for inference (matches inference_output_*_0.4)
IOU_THRES  = 0.5        # NMS IoU
N_BASELINE = 6          # how many synthetic test trays to show in the Cell 4 baseline

# Class names are read from the COCO categories at conversion time (source of truth);
# this list is only used for a cross-check.  Must match the COCO export order.
EXPECTED_CLASS_NAMES = [
    # Insecta — 17 orders.  Siphonaptera (6 specimens) and Raphidioptera (2) were
    # dropped from the taxonomy; their specimens are on the prune blocklist so they
    # never reached a tray.
    "Lepidoptera", "Psocodea", "Odonata", "Zoraptera",
    "Phasmatodea", "Zygentoma", "Thysanoptera", "Strepsiptera",
    "Archaeognatha", "Dermaptera",
    "Diptera", "Coleoptera", "Hymenoptera", "Hemiptera",
    "Orthoptera", "Trichoptera", "Neuroptera",
    # Arachnida — 2 orders
    "Araneae", "Opiliones",
    # Myriapods at ORDER level.  Previously the class-level labels "Diplopoda" and
    # "Chilopoda", each covering morphologically unrelated orders — which is why both
    # scored badly (0.19 and 0.34).
    "Polyxenida", "Lithobiomorpha", "Scutigeromorpha",
]

os.environ.setdefault("CUDA_VISIBLE_DEVICES", DEVICE)
print(f"COCO source : {COCO_DATASET_DIR}")
print(f"YOLO target : {YOLO_DATASET_DIR}")
print(f"flatbug init: {FLATBUG_WEIGHTS}  (arch {BASE_ARCH})")
print(f"Run         : {OUTPUT_DIR / RUN_NAME}")


---
## Cell 3 — Download the flatbug Weights

Downloads `flat_bug_<SIZE>.pt` from flatbug's public store if not already on disk.
These are the arthropod-fine-tuned YOLOv8-seg checkpoints; training uses only their
backbone + neck, while the pretrained baselines use the full model.

| Size | Backbone | Notes |
|---|---|---|
| `S` | yolov8s-seg | fastest / lowest VRAM |
| `M` | yolov8m-seg | **recommended** balance |
| `L` | yolov8l-seg | best features, most VRAM |


In [ ]:
import urllib.request
from pathlib import Path

FLATBUG_WEIGHTS = Path(FLATBUG_WEIGHTS)
if FLATBUG_WEIGHTS.exists():
    print(f"Already present: {FLATBUG_WEIGHTS}  ({FLATBUG_WEIGHTS.stat().st_size/1e6:.1f} MB)")
else:
    url = FLATBUG_REMOTE + FLATBUG_WEIGHTS.name
    print(f"Downloading {url}")
    tmp = FLATBUG_WEIGHTS.with_suffix(".pt.tmp")
    try:
        def _hook(b, bs, ts):
            if ts > 0:
                pct = min(100, 100 * b * bs / ts)
                print(f"\r  {pct:5.1f}%  ({b*bs/1e6:7.1f} MB)", end="")
        urllib.request.urlretrieve(url, tmp, reporthook=_hook)
        tmp.rename(FLATBUG_WEIGHTS)
        print(f"\nSaved -> {FLATBUG_WEIGHTS}  ({FLATBUG_WEIGHTS.stat().st_size/1e6:.1f} MB)")
    finally:
        if tmp.exists():
            tmp.unlink()


---
## Cell 4 — Baseline: Pretrained flatbug on Synthetic Trays (**before** training)

Runs the raw pretrained flatbug over a few synthetic test trays so you can see how it
behaves out-of-the-box. This is a **single-class** arthropod detector: it *localises*
insects but does **not** name orders — naming the 23 orders is what fine-tuning adds.

Primary path uses flatbug's native **tiled multi-scale** `Predictor`. If the `flat_bug`
package isn't importable, it falls back to plain Ultralytics seg-predict (no tiling).


In [ ]:
import torch
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

BASELINE_DIR = Path(OUTPUT_DIR) / "baseline_pretrained_flatbug"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

_test_dir = Path(COCO_DATASET_DIR) / "test"
baseline_imgs = sorted(list(_test_dir.glob("*.jpg")) + list(_test_dir.glob("*.png")))[:N_BASELINE]
assert baseline_imgs, f"No test images found in {_test_dir}"
print(f"Baseline: {len(baseline_imgs)} synthetic test trays through the *pretrained* flatbug (single-class).")

device = "cuda" if torch.cuda.is_available() else "cpu"
vis_paths, titles = [], []

use_native = True
try:
    from flat_bug.predictor import Predictor
except Exception as e:
    use_native = False
    print(f"flat_bug not importable ({e.__class__.__name__}); using Ultralytics fallback (no pyramid tiling).")

if use_native:
    dtype = torch.float16 if device == "cuda" else torch.float32
    fb = Predictor(str(FLATBUG_WEIGHTS), device=device, dtype=dtype)
    for img_path in baseline_imgs:
        res = fb.pyramid_predictions(str(img_path))
        out_png = BASELINE_DIR / f"baseline_{img_path.stem}.png"
        res.plot(outpath=str(out_png), masks=True, boxes=True, confidence=True, wait=True)
        vis_paths.append(out_png)
        titles.append(f"{img_path.name}  ({len(res)} det)")
else:
    from ultralytics import YOLO
    fb = YOLO(str(FLATBUG_WEIGHTS))
    preds = fb.predict(source=[str(p) for p in baseline_imgs], imgsz=IMG_SIZE,
                       device=DEVICE, conf=0.25, verbose=False)
    for img_path, res in zip(baseline_imgs, preds):
        out_png = BASELINE_DIR / f"baseline_{img_path.stem}.png"
        cv2.imwrite(str(out_png), res.plot(line_width=2))
        vis_paths.append(out_png)
        titles.append(f"{img_path.name}  ({len(res.boxes)} det)")

n = len(vis_paths); cols = 3; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
for ax, vp, t in zip(axes_flat, vis_paths, titles):
    ax.imshow(cv2.cvtColor(cv2.imread(str(vp)), cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=9)
    ax.axis("off")
for ax in axes_flat[n:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(BASELINE_DIR / "baseline_grid.png", dpi=110, bbox_inches="tight")
plt.show()
print("\nNOTE: pretrained flatbug is a SINGLE-CLASS arthropod detector - it finds insects but")
print("      cannot name orders. Fine-tuning below adds the 22-class labels.")


---
## Cell 5 — Real-Data Config: `3DTRAB_PITFALL_TRAP_Dataset_v1`

Sets up inference on the **real** pitfall-trap images (no ground truth → inference +
visualisation only, like your earlier real-tray dumps). Set `REAL_DATASET_DIR` to the
dataset's path on Ares; images are globbed **recursively**, so nested sub-folders are
fine. Both the pre-inference (Cell 6, before training) and post-inference (Cell 13,
after training) use what this cell defines.


In [ ]:
from pathlib import Path

# ← EDIT: path to the real dataset on Ares (recursively globbed for images).
REAL_DATASET_DIR = BASE_DIR / "3DTRAB_PITFALL_TRAP_Dataset_v1"

# Fine-tuned model for the POST run (Cell 13) — deterministic run path from Cell 10
# (works even after a kernel restart; point elsewhere to reuse another checkpoint).
POST_WEIGHTS = Path(OUTPUT_DIR) / RUN_NAME / "weights" / "best.pt"

REAL_MAX_IMAGES = None        # cap #images for a quick look (None = all). Native flatbug tiling is slow → try 30 first.
REAL_CONF_THRES = CONF_THRES  # 0.4, matching your real-tray dumps
REAL_GRID_N     = 12          # images shown in each annotated grid

_exts = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
REAL_DATASET_DIR = Path(REAL_DATASET_DIR)
assert REAL_DATASET_DIR.exists(), f"REAL_DATASET_DIR not found: {REAL_DATASET_DIR} (set the real Ares path)"
# Recursive glob, but skip hidden dirs/files (e.g. .ipynb_checkpoints) so we don't
# process duplicate checkpoint copies.
real_imgs = sorted(p for p in REAL_DATASET_DIR.rglob("*")
                   if p.suffix.lower() in _exts
                   and not any(part.startswith(".") for part in p.relative_to(REAL_DATASET_DIR).parts))
if REAL_MAX_IMAGES:
    real_imgs = real_imgs[:REAL_MAX_IMAGES]
assert real_imgs, f"No images found under {REAL_DATASET_DIR} (checked {_exts})"

REAL_OUT = Path(OUTPUT_DIR) / f"real_3DTRAB_pitfall_{REAL_CONF_THRES}"
REAL_OUT.mkdir(parents=True, exist_ok=True)
print(f"{len(real_imgs)} real images under {REAL_DATASET_DIR}")
print(f"Outputs -> {REAL_OUT}")


---
## Cell 6 — Real Data **PRE** Inference (pretrained flatbug, **before** training)

The raw arthropod detector on the real trap images: single class, so this measures
**localisation** (does it find the bugs?) not naming. Uses flatbug's native tiled
`Predictor` when available, else the Ultralytics seg-predict fallback. **Clears old
outputs, then annotates every image** (saved to `pre_pretrained_flatbug/annotated/`)
plus a per-image count CSV. **Needs only the downloaded weights — no training
required**, so run it before kicking off the long training below.


In [ ]:
import torch, cv2, csv, shutil
import numpy as np
from pathlib import Path

# Clear old outputs so nothing stale lingers, then annotate EVERY image (full set).
PRE_DIR = REAL_OUT / "pre_pretrained_flatbug"
shutil.rmtree(PRE_DIR, ignore_errors=True)
ANNOT_DIR = PRE_DIR / "annotated"
ANNOT_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

use_native = True
try:
    from flat_bug.predictor import Predictor
except Exception as e:
    use_native = False
    print(f"flat_bug not importable ({e.__class__.__name__}); using Ultralytics fallback (no pyramid tiling).")

counts = []
csv_path = PRE_DIR / "pre_detections.csv"
print(f"Annotating all {len(real_imgs)} real images -> {ANNOT_DIR}")
with open(csv_path, "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["image", "n_detections"])
    if use_native:
        dtype = torch.float16 if device == "cuda" else torch.float32
        fb = Predictor(str(FLATBUG_WEIGHTS), device=device, dtype=dtype)
        for i, img_path in enumerate(real_imgs):
            res = fb.pyramid_predictions(str(img_path))
            n = len(res); counts.append(n); wr.writerow([img_path.name, n])
            res.plot(outpath=str(ANNOT_DIR / f"{img_path.stem}.png"),
                     masks=True, boxes=True, confidence=True, wait=True)
            if (i + 1) % 20 == 0: print(f"  {i+1}/{len(real_imgs)}")
    else:
        from ultralytics import YOLO
        fb = YOLO(str(FLATBUG_WEIGHTS))
        for i, res in enumerate(fb.predict(source=[str(p) for p in real_imgs], imgsz=IMG_SIZE,
                                           device=DEVICE, conf=0.25, stream=True, verbose=False)):
            name = Path(res.path).name; n = len(res.boxes); counts.append(n); wr.writerow([name, n])
            cv2.imwrite(str(ANNOT_DIR / f"{Path(res.path).stem}.png"), res.plot(line_width=2))
            if (i + 1) % 20 == 0: print(f"  {i+1}/{len(real_imgs)}")

print(f"\nPretrained flatbug: {len(counts)} images, {sum(counts)} total detections, "
      f"mean {np.mean(counts):.1f}/img (median {int(np.median(counts))})")
print(f"Per-image counts CSV -> {csv_path}")
print(f"Annotated images      -> {ANNOT_DIR}  ({len(list(ANNOT_DIR.glob('*.png')))} files)")


---
## Cell 7 — Convert COCO → YOLO Detection Format

Ultralytics wants `images/{train,val,test}` + `labels/{train,val,test}` (one `.txt`
per image, `class cx cy w h` normalised) plus a `data.yaml`. We:
- read the class list straight from the COCO `categories` (source of truth),
- remap `category_id` → contiguous `0..nc-1`,
- **symlink** images (no multi-GB copy; falls back to copy if symlinks fail),
- map the COCO `valid/` split → YOLO `val/`.


In [ ]:
import json, os, shutil
from pathlib import Path
import yaml

COCO_DATASET_DIR = Path(COCO_DATASET_DIR)
YOLO_DATASET_DIR = Path(YOLO_DATASET_DIR)
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}   # coco name -> yolo name

def _load_coco(split):
    p = COCO_DATASET_DIR / split / "_annotations.coco.json"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p} - is COCO_DATASET_DIR correct?")
    return json.load(open(p))

# --- class order from the train split's categories (id-sorted) ---------------
train_coco = _load_coco("train")
cats = sorted(train_coco["categories"], key=lambda c: c["id"])
CLASS_NAMES = [c["name"] for c in cats]
id2idx = {c["id"]: i for i, c in enumerate(cats)}
NC = len(CLASS_NAMES)
print(f"{NC} classes from COCO categories:\n  {CLASS_NAMES}")
if list(CLASS_NAMES) != list(EXPECTED_CLASS_NAMES):
    print("\nWARNING: class order differs from EXPECTED_CLASS_NAMES in the config cell!")
    print("         Using the COCO order above (authoritative).")

def _link_or_copy(src, dst):
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(os.path.abspath(src), dst)
    except OSError:
        shutil.copy2(src, dst)

def convert_split(coco_split, yolo_split):
    coco = _load_coco(coco_split)
    src_dir = COCO_DATASET_DIR / coco_split
    img_out = YOLO_DATASET_DIR / "images" / yolo_split
    lbl_out = YOLO_DATASET_DIR / "labels" / yolo_split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    imgs = {im["id"]: im for im in coco["images"]}
    anns_by_img = {}
    for a in coco["annotations"]:
        anns_by_img.setdefault(a["image_id"], []).append(a)

    n_ann = 0
    for iid, im in imgs.items():
        fn, W, H = im["file_name"], im["width"], im["height"]
        _link_or_copy(src_dir / fn, img_out / fn)
        lines = []
        for a in anns_by_img.get(iid, []):
            x, y, w, h = a["bbox"]
            cx = min(max((x + w / 2) / W, 0.0), 1.0)
            cy = min(max((y + h / 2) / H, 0.0), 1.0)
            nw = min(max(w / W, 0.0), 1.0)
            nh = min(max(h / H, 0.0), 1.0)
            lines.append(f"{id2idx[a['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            n_ann += 1
        (lbl_out / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
    return len(imgs), n_ann

counts = {}
for coco_split, yolo_split in SPLIT_MAP.items():
    ni, na = convert_split(coco_split, yolo_split)
    counts[yolo_split] = (ni, na)
    print(f"  {coco_split:5s} -> {yolo_split:5s}: {ni:5d} images, {na:6d} annotations")

# --- data.yaml ----------------------------------------------------------------
data_yaml = {
    "path": str(YOLO_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": NC,
    "names": CLASS_NAMES,
}
DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(f"\nWrote {DATA_YAML_PATH}")


---
## Cell 8 — Sanity-Check the Converted Dataset

Confirms image/label counts line up, per-class instance histogram, and that a
random label file looks well-formed (classes in range, coords in [0,1]).


In [ ]:
from pathlib import Path
import random
from collections import Counter

random.seed(SEED)
YOLO_DATASET_DIR = Path(YOLO_DATASET_DIR)

for yolo_split in ["train", "val", "test"]:
    imgs = list((YOLO_DATASET_DIR / "images" / yolo_split).glob("*"))
    lbls = list((YOLO_DATASET_DIR / "labels" / yolo_split).glob("*.txt"))
    assert len(imgs) == len(lbls), f"{yolo_split}: {len(imgs)} images vs {len(lbls)} labels!"
    hist = Counter()
    bad = 0
    for lf in lbls:
        for line in lf.read_text().splitlines():
            if not line.strip():
                continue
            c, cx, cy, w, h = line.split()
            c = int(c)
            hist[c] += 1
            if not (0 <= c < NC and all(0.0 <= float(v) <= 1.0 for v in (cx, cy, w, h))):
                bad += 1
    total = sum(hist.values())
    print(f"\n[{yolo_split}] {len(imgs)} images, {total} instances, malformed={bad}")
    for cid in range(NC):
        if hist.get(cid, 0):
            print(f"    {cid:2d} {CLASS_NAMES[cid]:<14s} {hist[cid]:6d}")

sample = random.choice(list((YOLO_DATASET_DIR / "labels" / "train").glob("*.txt")))
print(f"\nSample label {sample.name}:")
print("\n".join(sample.read_text().splitlines()[:5]) or "  (empty / no objects)")


---
## Cell 9 — Build Model & Warm-Start From flatbug

We load the matching COCO-pretrained YOLOv8 **detection** model, then transfer
flatbug's arthropod-tuned weights on top with `model.load()`. Ultralytics
intersects state-dicts by name **and shape**, so flatbug's backbone + neck are
copied in and its single-class *segmentation* head is skipped. The detection head
is then reset to our 22 classes when training starts.


In [ ]:
from ultralytics import YOLO

# COCO-pretrained detection model of the matching scale (backbone/neck shapes match flatbug).
model = YOLO(BASE_ARCH)

# Transfer flatbug's arthropod-tuned backbone + neck (segmentation head skipped on shape mismatch).
model.load(str(FLATBUG_WEIGHTS))
print(f"Warm-started {BASE_ARCH} from flatbug weights: {FLATBUG_WEIGHTS}")
print("(Detection head is reset to the 22 dataset classes at train start.)")


---
## Cell 10 — Train

Fine-tunes on the 22-class trays. Early-stops after `PATIENCE` epochs without
improvement. Checkpoints + curves land in `OUTPUT_DIR/RUN_NAME/`.


In [ ]:
from pathlib import Path
import sys
from ultralytics import settings

# Ultralytics auto-enables optional logging callbacks when their packages are present.
# The newer MLflow in this shared env refuses its default filesystem tracking store and
# crashes training at setup ("filesystem tracking backend ... is in maintenance mode").
# Disable the cloud loggers permanently, and drop any already-cached logger module so
# the change also applies in an already-running kernel (no restart needed).
settings.update({"mlflow": False, "wandb": False, "comet": False,
                 "dvc": False, "clearml": False, "neptune": False})
for _m in [k for k in list(sys.modules) if "ultralytics.utils.callbacks.mlflow" in k]:
    del sys.modules[_m]

train_results = model.train(
    data=str(Path(YOLO_DATASET_DIR) / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    close_mosaic=CLOSE_MOSAIC,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    project=str(OUTPUT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    val=True,
)

# model.trainer.{save_dir,best} are the canonical, version-stable handles.
RUN_DIR      = Path(model.trainer.save_dir)
BEST_WEIGHTS = Path(model.trainer.best)          # <run>/weights/best.pt
assert BEST_WEIGHTS.exists(), f"best.pt not found at {BEST_WEIGHTS}"
print(f"\nRun dir      : {RUN_DIR}")
print(f"Best weights : {BEST_WEIGHTS}")


## OPTION — Two-stage transfer: real flat-bug detection → synthetic

Run these **instead of** the single-stage warm-start + train cells above. **Stage 1** pretrains on the real flat-bug arthropod images (1-class *detection*) → **Stage 2** fine-tunes on the synthetic 22-class trays. The real localisation prior transfers via the backbone/neck; the head is re-sized 1→23 automatically. Result `BEST_WEIGHTS` feeds the existing test-eval + inference cells.


In [ ]:
from pathlib import Path

# ============================================================
# TWO-STAGE config — real flat-bug detection pretrain -> synthetic
# ============================================================
# Stage 1 pretrains on the REAL flat-bug arthropod dataset (1-class 'insect'
# DETECTION) to give a real-world localisation prior, then Stage 2 fine-tunes on
# the mixed 22-class trays for order classification.
#
# Prepare the flat-bug data first (on the server):
#   s5cmd --endpoint-url https://se-sto-1.linodeobjects.com sync \
#         "s3://flat-bug-data-stable/pre-pro/*" ./pre-pro     # (needs AWS keys; or Zenodo 10.5281/zenodo.14761446)
#   fb_prepare_data -i ./pre-pro -o ./fb_yolo                 # -> YOLO data.yaml (1-class)
FB_DET_YAML   = str(BASE_DIR / "fb_yolo" / "data.yaml")   # <-- point at your prepared flat-bug YOLO data
STAGE1_NAME   = RUN_NAME + "_stage1_flatbug"
STAGE1_EPOCHS = 50                                        # real-data localisation pretrain (Stage 2 uses EPOCHS)
assert Path(FB_DET_YAML).exists(), f"flat-bug YOLO data.yaml not found: {FB_DET_YAML} (run fb_prepare_data first)"
print(f"Stage 1 data : {FB_DET_YAML}  ({STAGE1_EPOCHS} ep)")
print(f"Stage 2 data : {YOLO_DATASET_DIR}/data.yaml  ({EPOCHS} ep)")


In [ ]:
# ── STAGE 1 — pretrain on real flat-bug data (1-class 'insect' detection) ──
from ultralytics import YOLO, settings
import sys
settings.update({"mlflow": False, "wandb": False, "comet": False,
                 "dvc": False, "clearml": False, "neptune": False})
for _m in [k for k in list(sys.modules) if "ultralytics.utils.callbacks.mlflow" in k]:
    del sys.modules[_m]

# arthropod-tuned backbone/neck from flatbug (same warm-start as the single-stage run)
s1 = YOLO(BASE_ARCH)
s1.load(str(FLATBUG_WEIGHTS))
print(f"Stage 1: warm-started {BASE_ARCH} from {FLATBUG_WEIGHTS}; training on real flat-bug data.")

s1.train(
    data=FB_DET_YAML, epochs=STAGE1_EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=DEVICE,
    workers=WORKERS, optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
    patience=PATIENCE, seed=SEED, deterministic=True,
    project=str(OUTPUT_DIR), name=STAGE1_NAME, exist_ok=True, plots=True, val=True,
)
S1_BEST = Path(s1.trainer.best)
assert S1_BEST.exists(), f"Stage-1 best.pt not found: {S1_BEST}"
print(f"\nStage-1 best (real-arthropod detector): {S1_BEST}")


In [ ]:
# ── STAGE 2 — fine-tune the flat-bug-pretrained detector on the synthetic 22 classes ──
# Ultralytics auto-resizes the detection head from 1 -> 22 classes when the new
# data.yaml has 22 names. Everything else (backbone/neck) carries the real prior.
from ultralytics import YOLO

s2 = YOLO(str(S1_BEST))
print(f"Stage 2: fine-tuning {S1_BEST.name} on mixed 22-class trays.")

train_results = s2.train(
    data=str(Path(YOLO_DATASET_DIR) / "data.yaml"), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
    device=DEVICE, workers=WORKERS, optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
    patience=PATIENCE, seed=SEED, deterministic=True,
    project=str(OUTPUT_DIR), name=RUN_NAME, exist_ok=True, plots=True, val=True,
)
RUN_DIR      = Path(s2.trainer.save_dir)
BEST_WEIGHTS = Path(s2.trainer.best)   # downstream test/inference cells use this
model = s2
assert BEST_WEIGHTS.exists(), f"Stage-2 best.pt not found: {BEST_WEIGHTS}"
print(f"\nStage-2 run dir : {RUN_DIR}")
print(f"Stage-2 best    : {BEST_WEIGHTS}")


---
## Cell 11 — Evaluate on the Held-Out Test Split

Overall mAP50-95 / mAP50 and a per-class mAP50-95 table — directly comparable to
the RF-DETR test numbers.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

# Resolve the checkpoint from DISK, not from a variable set earlier in the session.
# BEST_WEIGHTS is only bound in memory by the training cell; after a detached run the
# kernel is usually fresh, and this cell — the sole source of flatbug's test mAP75,
# precision and recall — would die with NameError.
_cands = sorted(Path(OUTPUT_DIR).glob("*/weights/best.pt")) or \
         sorted(Path(OUTPUT_DIR).glob("**/weights/best.pt"))
try:
    BEST_WEIGHTS
except NameError:
    assert _cands, f"no best.pt under {OUTPUT_DIR} — has training finished?"
    BEST_WEIGHTS = _cands[-1]
    print(f"BEST_WEIGHTS was not in memory; resolved from disk: {BEST_WEIGHTS}")

best = YOLO(str(BEST_WEIGHTS))
test_metrics = best.val(
    data=str(Path(YOLO_DATASET_DIR) / "data.yaml"),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    plots=True,
    project=str(OUTPUT_DIR),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)

print(f"\n=== TEST metrics ({RUN_NAME}) ===")
print(f"  mAP50-95 : {test_metrics.box.map:.4f}")
print(f"  mAP50    : {test_metrics.box.map50:.4f}")
print(f"  mAP75    : {test_metrics.box.map75:.4f}")
print(f"  precision: {test_metrics.box.mp:.4f}")
print(f"  recall   : {test_metrics.box.mr:.4f}")

# PERSIST THEM. Ultralytics' val() writes plots and nothing else — no CSV, no JSON
# unless save_json is set. Without this the whole flatbug half of the comparison
# lives only in this cell's stdout, and recovering it means another GPU run.
_names = best.names
_maps  = test_metrics.box.maps
_TESTDIR = Path(OUTPUT_DIR) / f"{RUN_NAME}_test"
_TESTDIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame([{
    "model": RUN_NAME, "split": "test", "imgsz": IMG_SIZE,
    "mAP50_95": round(float(test_metrics.box.map), 4),
    "mAP50":    round(float(test_metrics.box.map50), 4),
    "mAP75":    round(float(test_metrics.box.map75), 4),
    "precision": round(float(test_metrics.box.mp), 4),
    "recall":    round(float(test_metrics.box.mr), 4),
}]).to_csv(_TESTDIR / "test_metrics.csv", index=False)

_pc = []
for cid in range(len(_names)):
    row = {"class_id": cid, "class": _names[cid],
           "mAP50_95": round(float(_maps[cid]), 4) if cid < len(_maps) else None}
    # per-class P/R/AP50 when ultralytics exposes them
    for attr, col in (("p","precision"), ("r","recall"), ("ap50","AP50")):
        v = getattr(test_metrics.box, attr, None)
        if v is not None and cid < len(v): row[col] = round(float(v[cid]), 4)
    # per-class AP@75. box.maps is mAP@50:95 only, but box.all_ap holds the full
    # (classes x 10) IoU sweep from 0.50 to 0.95, so column 5 IS AP@75. Without this
    # the comparison has no per-class AP75 on the flatbug side, and AP75 is the
    # metric that shows whether boxes are tight.
    _aa = getattr(test_metrics.box, "all_ap", None)
    if _aa is not None and cid < len(_aa) and len(_aa[cid]) > 5:
        row["AP75"] = round(float(_aa[cid][5]), 4)
    _pc.append(row)
_pcdf = pd.DataFrame(_pc)
_pcdf.to_csv(_TESTDIR / "per_class_metrics.csv", index=False)

print(f"\nwritten: {_TESTDIR/'test_metrics.csv'}")
print(f"         {_TESTDIR/'per_class_metrics.csv'}")
print("\nPer-class:")
print(_pcdf.to_string(index=False))


---
## Cell 12 — Post-Training Inference on Synthetic Test, Write `detections.csv`

Runs the fine-tuned model over every synthetic test image at `CONF_THRES`, writes a
`detections.csv` (schema `image,class,confidence,x1,y1,x2,y2,width,height` — same as
the RF-DETR dumps) and an annotated grid. Compare against the Cell 4 baseline.


In [ ]:
# `best` comes from the cell above; re-resolve it if this is a fresh kernel.
try:
    best
except NameError:
    from ultralytics import YOLO
    from pathlib import Path as _P
    _c = sorted(_P(OUTPUT_DIR).glob("*/weights/best.pt")) or \
         sorted(_P(OUTPUT_DIR).glob("**/weights/best.pt"))
    assert _c, f"no best.pt under {OUTPUT_DIR}"
    best = YOLO(str(_c[-1])); print(f"resolved best from disk: {_c[-1]}")

from pathlib import Path
import csv
import matplotlib.pyplot as plt
import cv2

INFER_DIR = Path(OUTPUT_DIR) / f"{RUN_NAME}_inference_{CONF_THRES}"
INFER_DIR.mkdir(parents=True, exist_ok=True)

test_imgs = sorted((Path(YOLO_DATASET_DIR) / "images" / "test").glob("*"))
print(f"Running inference on {len(test_imgs)} test images at conf>={CONF_THRES} ...")

csv_path = INFER_DIR / "detections.csv"
n_det = 0
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image", "class", "confidence", "x1", "y1", "x2", "y2", "width", "height"])
    for res in best.predict(source=[str(p) for p in test_imgs], conf=CONF_THRES,
                            iou=IOU_THRES, imgsz=IMG_SIZE, device=DEVICE,
                            stream=True, verbose=False):
        img_name = Path(res.path).name
        for b in res.boxes:
            cls = best.names[int(b.cls)]
            conf = float(b.conf)
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
            w.writerow([img_name, cls, f"{conf:.4f}", f"{x1:.1f}", f"{y1:.1f}",
                        f"{x2:.1f}", f"{y2:.1f}", f"{x2-x1:.1f}", f"{y2-y1:.1f}"])
            n_det += 1
print(f"Wrote {n_det} detections -> {csv_path}")

# --- annotated grid of first 12 test images ---------------------------------
grid_imgs = test_imgs[:12]
preds = best.predict(source=[str(p) for p in grid_imgs], conf=CONF_THRES,
                     iou=IOU_THRES, imgsz=IMG_SIZE, device=DEVICE, verbose=False)
cols, rows = 4, 3
fig, axes = plt.subplots(rows, cols, figsize=(20, 15))
for ax, res in zip(axes.flat, preds):
    ax.imshow(cv2.cvtColor(res.plot(line_width=2), cv2.COLOR_BGR2RGB))
    ax.set_title(Path(res.path).name, fontsize=8)
    ax.axis("off")
for ax in axes.flat[len(preds):]:
    ax.axis("off")
plt.tight_layout()
grid_path = INFER_DIR / "test_inference_grid.png"
plt.savefig(grid_path, dpi=110, bbox_inches="tight")
print(f"Saved grid -> {grid_path}")
plt.show()


---
## Cell 13 — Real Data **POST** Inference (fine-tuned 22-class model, **after** training)

The fine-tuned model on the same real images from Cell 5. **Clears old outputs, then
annotates every image** (saved to `post_finetuned/annotated/`), writes `detections.csv`
(same schema as your RF-DETR dumps), and prints the **class distribution**. Watch whether
it funnels detections into size-based catch-alls (Thysanoptera / Araneae) the way RF-DETR
did — flatbug already cut Thysanoptera 48.9%→8.3% and Araneae 26.9%→13.0%, so use the
annotated images to judge how many of the remaining Araneae boxes are genuinely spiders.


In [ ]:
from ultralytics import YOLO
import csv, cv2, shutil
from pathlib import Path
from collections import Counter

POST_WEIGHTS = Path(POST_WEIGHTS)
assert POST_WEIGHTS.exists(), f"POST_WEIGHTS not found: {POST_WEIGHTS} - run training (Cell 10) or set it to a checkpoint."
post = YOLO(str(POST_WEIGHTS))

# Clear old outputs, then annotate EVERY image (full set) + dump detections.csv.
POST_DIR = REAL_OUT / "post_finetuned"
shutil.rmtree(POST_DIR, ignore_errors=True)
ANNOT_DIR = POST_DIR / "annotated"
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = POST_DIR / "detections.csv"
cls_hist = Counter(); n_det = 0
print(f"Annotating all {len(real_imgs)} real images -> {ANNOT_DIR}")
with open(csv_path, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["image", "class", "confidence", "x1", "y1", "x2", "y2", "width", "height"])
    for i, res in enumerate(post.predict(source=[str(p) for p in real_imgs], conf=REAL_CONF_THRES,
                                         iou=IOU_THRES, imgsz=IMG_SIZE, device=DEVICE,
                                         stream=True, verbose=False)):
        name = Path(res.path).name
        cv2.imwrite(str(ANNOT_DIR / f"{Path(res.path).stem}.png"), res.plot(line_width=2))   # annotate every image
        for b in res.boxes:
            cls = post.names[int(b.cls)]; conf = float(b.conf)
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
            wr.writerow([name, cls, f"{conf:.4f}", f"{x1:.1f}", f"{y1:.1f}",
                         f"{x2:.1f}", f"{y2:.1f}", f"{x2-x1:.1f}", f"{y2-y1:.1f}"])
            cls_hist[cls] += 1; n_det += 1
        if (i + 1) % 20 == 0: print(f"  {i+1}/{len(real_imgs)}")

print(f"\nFine-tuned model: {n_det} detections over {len(real_imgs)} images "
      f"({n_det/max(len(real_imgs),1):.1f}/img)")
print(f"detections.csv   -> {csv_path}")
print(f"Annotated images -> {ANNOT_DIR}  ({len(list(ANNOT_DIR.glob('*.png')))} files)")
print("\nClass distribution (compare vs RF-DETR: was 48.9% Thysanoptera + 26.9% Araneae):")
for cls, c in cls_hist.most_common():
    print(f"  {cls:<14s} {c:5d}  ({100*c/max(n_det,1):4.1f}%)")


---
## Cell 13b — Real Data POST Inference at a **lower threshold (0.2)** — diagnostic

Re-runs the **fine-tuned** model on the real images at **conf 0.2** (vs 0.4 in Cell 13)
to test one thing: are the **missed dense-cluster specimens** proposed-but-suppressed
(they show up here) or never proposed (still missing)? Writes to a **separate**
`real_3DTRAB_pitfall_0.2/` folder, so the 0.4 run in Cell 13 is left untouched.

**Runs without training** — it loads `best.pt` from disk. On a fresh kernel just run:
**Cell 1** (install, once per kernel) -> **Cell 2** (config) -> **Cell 5** (real config)
-> **this cell**. You do *not* need Cells 7-12. Keep the RF-DETR comparison at 0.4 for
both models; read this 0.2 pass only as a diagnostic (it will also surface more debris
false positives, so the totals will look noisier — that's expected).


In [ ]:
from ultralytics import YOLO
import csv, cv2, shutil
from pathlib import Path
from collections import Counter

CONF_LOW = 0.1   # ← lower confidence: test if missed cluster specimens are suppressed vs not proposed

# Loads best.pt from disk -> NO training / no in-memory model needed (run Cell 2 + Cell 5 first).
POST_WEIGHTS = Path(POST_WEIGHTS)
assert POST_WEIGHTS.exists(), f"POST_WEIGHTS not found: {POST_WEIGHTS} - train (Cell 10) or point it at a checkpoint."
assert "real_imgs" in globals() and real_imgs, "real_imgs is empty - run Cell 5 (real config) first."
post = YOLO(str(POST_WEIGHTS))

# Separate output folder (threshold-suffixed) so the 0.4 run is NOT overwritten.
REAL_OUT_LOW = Path(OUTPUT_DIR) / f"real_3DTRAB_pitfall_{CONF_LOW}"
POST_DIR = REAL_OUT_LOW / "post_finetuned"
shutil.rmtree(POST_DIR, ignore_errors=True)
ANNOT_DIR = POST_DIR / "annotated"
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = POST_DIR / "detections.csv"
cls_hist = Counter(); n_det = 0
print(f"[conf={CONF_LOW}] annotating all {len(real_imgs)} real images -> {ANNOT_DIR}")
with open(csv_path, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["image", "class", "confidence", "x1", "y1", "x2", "y2", "width", "height"])
    for i, res in enumerate(post.predict(source=[str(p) for p in real_imgs], conf=CONF_LOW,
                                         iou=IOU_THRES, imgsz=IMG_SIZE, device=DEVICE,
                                         stream=True, verbose=False)):
        name = Path(res.path).name
        cv2.imwrite(str(ANNOT_DIR / f"{Path(res.path).stem}.png"), res.plot(line_width=2))
        for b in res.boxes:
            cls = post.names[int(b.cls)]; conf = float(b.conf)
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
            wr.writerow([name, cls, f"{conf:.4f}", f"{x1:.1f}", f"{y1:.1f}",
                         f"{x2:.1f}", f"{y2:.1f}", f"{x2-x1:.1f}", f"{y2-y1:.1f}"])
            cls_hist[cls] += 1; n_det += 1
        if (i + 1) % 20 == 0: print(f"  {i+1}/{len(real_imgs)}")

print(f"\n[conf={CONF_LOW}] {n_det} detections over {len(real_imgs)} images ({n_det/max(len(real_imgs),1):.1f}/img)")
print(f"detections.csv   -> {csv_path}")
print(f"Annotated images -> {ANNOT_DIR}  ({len(list(ANNOT_DIR.glob('*.png')))} files)")
print(f"\nClass distribution @ conf={CONF_LOW} (compare vs the 0.4 run in Cell 13):")
for cls, c in cls_hist.most_common():
    print(f"  {cls:<14s} {c:5d}  ({100*c/max(n_det,1):4.1f}%)")
print("\nDIAGNOSTIC -> open the SAME cluster images as the 0.4 run")
print("  (e.g. Overview_Group6_Sample1_ET2_TIME1428_0080, Overview_Group1TP_Sample1_ET2_TIME1047_0000)")
print("  and check if the previously-MISSED central cluster now gets boxes:")
print("    specimens appear  -> threshold issue (tunable)")
print("    still missing     -> not proposed by the model (needs sim2real, not a threshold change)")


---
## Cell 14 — Export to ONNX (optional, for deployment)


In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMG_SIZE, opset=12, dynamic=True, simplify=True)
print(f"Exported ONNX -> {onnx_path}")
